# Spendende für *CharityML* finden

In diesem Projekt testen wir mehrere Algorithmen für überwachtes Lernen, um das Einkommen von Personen anhand von Daten aus dem US-Zensus von 1994 treffsicher zu modellieren. Danach wählen wir anhand vorläufiger Ergebnisse den besten Kandidaten unter den Algorithmen aus und optimieren diesen Algorithmus weiter, um die Daten bestmöglich zu modellieren. Unser Ziel bei dieser Implementierung ist es, ein Modell zu konstruieren, das treffsicher vorhersagt, ob eine Person mehr als 50.000 $ verdient. Eine solche Aufgabe kann im gemeinnützigen Bereich auftreten, wo Organisationen von Spenden leben.  Das Einkommen einer Person zu kennen, kann einer gemeinnützigen Organisation helfen, besser einzuschätzen, wie hoch die Spende sein sollte, um die sie bittet, oder ob sie überhaupt Kontakt aufnehmen sollte.  Auch wenn es schwierig sein kann, die allgemeine Einkommensklasse einer Person direkt aus öffentlichen Quellen zu bestimmen, können wir diesen Wert aus anderen öffentlich verfügbaren Features ableiten.

Der Datensatz für dieses Projekt stammt aus dem [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Census+Income). Ron Kohavi und Barry Becker haben den Datensatz zur Verfügung gestellt, nachdem er im Artikel _„Scaling Up the Accuracy of Naive-Bayes Classifiers: A Decision-Tree Hybrid“_ veröffentlicht worden war. Du findest den Artikel von Ron Kohavi [online](https://aaai.org/papers/033-scaling-up-the-accuracy-of-naive-bayes-classifiers-a-decision-tree-hybrid/). Die Daten, die wir hier untersuchen, enthalten kleine Änderungen gegenüber dem ursprünglichen Datensatz, etwa das Entfernen des Features `'fnlwgt'` und von Einträgen mit fehlenden oder falsch formatierten Werten.

## Lernziele

Am Ende dieses Notebooks kannst du:

- gemischte Zensusdaten mit Log-Transformationen, One-Hot Encoding und Feature-Skalierung vorverarbeiten.
- eine naive Baseline festlegen und eine passende Evaluationsmetrik wählen (den F-beta-Score).
- einen Entscheidungsbaum und einen AdaBoost-Klassifikator anhand von Accuracy und F-Score vergleichen.
- das stärkste Modell mit Grid Search tunen und die Auswirkung eines verkleinerten Feature-Sets beurteilen.

----
## Daten erkunden

Beginnen wir damit, die nötigen Bibliotheken zu importieren, die Daten einzulesen und uns den Datensatz anzusehen.

Beachte, dass die letzte Spalte dieses Datensatzes, `'income'`, unsere Zielvariable sein wird (ob eine Person jährlich mehr als oder höchstens 50.000 $ verdient). Alle anderen Spalten sind Features zu jeder Person in der Zensus-Datenbank.

In [ ]:
# Für dieses Projekt benötigte Bibliotheken importieren
import numpy as np
import pandas as pd
from time import time

# Bibliotheken für die Visualisierung importieren
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(style="whitegrid", color_codes=True)

# Funktionen aus der eigenen Python-Datei importieren (siehe visuals_script.py im Repo)
import sys
sys.path.append("..")  # visuals_script.py liegt im Stammverzeichnis des Repos
import visuals_script as vs

# Schönere Anzeige in Notebooks
%matplotlib inline

Beginnen wir damit, den Datensatz zu importieren und die ersten Zeilen zu untersuchen. 

In [ ]:
# Zensus-Datensatz laden
data = pd.read_csv("../data/census.csv")

In [ ]:
# Datentypen untersuchen
data.info()

Eine einfache Untersuchung des Datensatzes kann zeigen, wie viele Personen in die eine oder die andere Gruppe fallen, und uns den Prozentsatz der Personen nennen, die mehr als 50.000 \$ verdienen.

Sehen wir uns Folgendes an:

- Die Gesamtzahl der Einträge, `'n_records'`
- Die Anzahl der Personen, die jährlich mehr als 50.000 \$ verdienen, `'n_greater_50k'`.
- Die Anzahl der Personen, die jährlich höchstens 50.000 \$ verdienen, `'n_at_most_50k'`.
- Der Prozentsatz der Personen, die jährlich mehr als 50.000 \$ verdienen, `'greater_percent'`.

In [ ]:
# TODO: Gesamtzahl der Einträge
n_records = len(data)

# TODO: Anzahl der Einträge, bei denen das Einkommen der Person über 50.000 $ liegt
n_greater_50k = len(data[data['income'] == '>50K'])

# TODO: Anzahl der Einträge, bei denen das Einkommen der Person höchstens 50.000 $ beträgt
n_at_most_50k = len(data[data['income'] == '<=50K'])

# TODO: Anteil der Personen mit einem Einkommen über 50.000 $ in Prozent
greater_percent = 100 * n_greater_50k / n_records

# Ergebnisse ausgeben
print ("Total number of records: {}".format(n_records))
print ("Individuals making more than $50,000: {}".format(n_greater_50k))
print ("Individuals making at most $50,000: {}".format(n_at_most_50k))
print ("Percentage of individuals making more than $50,000: {:.2f}%".format(greater_percent))

Wir können auch den Zusammenhang zwischen verschiedenen Features einer Person und ihrem Einkommen visualisieren.

Sehen wir uns an, wie sich die Anzahl der Menschen, die über oder unter 50K verdienen, nach Geschlecht und Bildungsniveau aufteilt.

In [ ]:
# Eine Möglichkeit, den Zusammenhang zu visualisieren, ist ein catplot von seaborn
sns.catplot(x="sex", col='education_level', data=data, hue='income', kind="count", col_wrap=4);

----
## Daten vorbereiten
Bevor Daten als Eingabe für Machine-Learning-Algorithmen verwendet werden können, müssen sie oft bereinigt, formatiert und umstrukturiert werden. Dieser Schritt heißt üblicherweise **Vorverarbeitung**. Glücklicherweise gibt es in diesem Datensatz keine ungültigen oder fehlenden Einträge, mit denen wir umgehen müssen, allerdings haben bestimmte Features einige Eigenschaften, die angepasst werden müssen. Diese Vorverarbeitung kann das Ergebnis und die Vorhersagekraft fast aller Lernalgorithmen enorm verbessern.

In [ ]:
# Bevor wir weitermachen, teilen wir die Daten in Features und Zielvariable auf
features = data.drop('income', axis = 1)
target = data['income']

### Schiefe stetige Features transformieren
Ein Datensatz kann manchmal mindestens ein Feature enthalten, dessen Werte meist nahe bei einer einzigen Zahl liegen, das aber auch eine nicht unerhebliche Anzahl von Werten hat, die weit größer oder kleiner als diese Zahl sind.  Algorithmen können empfindlich gegenüber solchen Werteverteilungen sein und schlechter abschneiden, wenn der Wertebereich nicht richtig normalisiert ist. Im Zensus-Datensatz passen zwei Features zu dieser Beschreibung: '`capital-gain'` und `'capital-loss'`.

Stellen wir ein Histogramm dieser beiden Features dar und sehen wir uns an, wie sie verteilt sind. (Schau dir das Python-Skript in diesem Repo an, wenn dich der Code für diese Plot-Funktion interessiert.)

In [ ]:
# Schiefe stetige Features der Originaldaten visualisieren
vs.distribution(data)

Bei stark schiefen Feature-Verteilungen wie `'capital-gain'` und `'capital-loss'` ist es gängige Praxis, eine <a href="https://en.wikipedia.org/wiki/Data_transformation_(statistics)">logarithmische Transformation</a> auf die Daten anzuwenden, damit die sehr großen und sehr kleinen Werte die Leistung eines Lernalgorithmus nicht negativ beeinflussen. Eine logarithmische Transformation verkleinert den Wertebereich, den Ausreißer verursachen, deutlich.

In [ ]:
# Log-Transformation der schiefen Features ['capital-gain', 'capital-loss']
skewed = ['capital-gain', 'capital-loss']
features[skewed] = features[skewed].apply(lambda x: np.log(x + 1))

# Neue logarithmierte Verteilungen visualisieren 
vs.distribution(features, transformed = True)

### Datenvorverarbeitung

In der Tabelle unter **Daten erkunden** oben sehen wir, dass es für jeden Eintrag mehrere nicht numerische Features gibt. Lernalgorithmen erwarten normalerweise numerische Eingaben, deshalb müssen nicht numerische Features (sogenannte *kategoriale Variablen*) umgewandelt werden. Eine beliebte Methode, kategoriale Variablen umzuwandeln, ist das **One-Hot Encoding**. Das One-Hot Encoding erzeugt für jede mögliche Kategorie jedes nicht numerischen Features eine _„Dummy“_-Variable.

Außerdem müssen wir, wie bei den nicht numerischen Features, die nicht numerische Zielvariable `'income'` in numerische Werte umwandeln, damit der Lernalgorithmus funktioniert. Da es für dieses Label nur zwei mögliche Kategorien gibt ("<=50K" und ">50K"), können wir auf One-Hot Encoding verzichten und diese beiden Kategorien einfach als `0` bzw. `1` kodieren.

In [ ]:
# TODO: Daten in 'features_raw' mit pandas.get_dummies() per One-Hot Encoding kodieren
features = pd.get_dummies(features)

# TODO: Daten in 'income'/'target' als numerische Werte kodieren
target = target.apply(lambda x: 1 if x == '>50K' else 0)

# Anzahl der Features nach dem One-Hot Encoding ausgeben
encoded = list(features.columns)
print ("{} total features after one-hot encoding.".format(len(encoded)))

### Daten mischen und aufteilen

Jetzt sind alle _kategorialen Variablen_ in numerische Features umgewandelt. Als Nächstes teilen wir die Daten (sowohl die Features als auch ihre Labels) in einen Trainings- und einen Testdatensatz auf. 80% der Daten verwenden wir für das Training und 20% für den Test.

In [ ]:
# train_test_split importieren
from sklearn.model_selection import train_test_split

# Daten aus 'features' und 'income' in Trainings- und Testdatensatz aufteilen
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.2, random_state = 0)

# Ergebnisse der Aufteilung anzeigen
print ("Training set has {} samples.".format(X_train.shape[0]))
print ("Testing set has {} samples.".format(X_test.shape[0]))

### Numerische Features normalisieren
Es empfiehlt sich oft, numerische Features in irgendeiner Form zu skalieren. Eine Skalierung ändert die Form der Verteilung eines Features nicht (etwa bei `'capital-gain'` oder `'capital-loss'` oben); die Normalisierung sorgt aber dafür, dass jedes Feature gleich behandelt wird, wenn Algorithmen für überwachtes Lernen angewendet werden. Beachte, dass die Daten nach der Skalierung in ihrer Rohform nicht mehr dieselbe ursprüngliche Bedeutung haben, wie das Beispiel unten zeigt. Um Datenlecks zu vermeiden, normalisieren wir die Daten erst, nachdem wir sie in Trainings- und Testdatensatz aufgeteilt haben.

In [ ]:
# sklearn.preprocessing.StandardScaler importieren
from sklearn.preprocessing import MinMaxScaler

# Scaler initialisieren und dann auf die Features anwenden
scaler = MinMaxScaler()
numerical = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
X_train[numerical] = scaler.fit_transform(X_train[numerical])
X_test[numerical] = scaler.transform(X_test[numerical])

# Beispiel eines Eintrags nach der Skalierung anzeigen
X_train[numerical].sample(1)

----
## Modellleistung evaluieren
In diesem Abschnitt untersuchen wir vier verschiedene Algorithmen und bestimmen, welcher die Daten am besten modelliert.

### Metriken und das naive Vorhersagemodell

*CharityML* weiß aus eigener Recherche, dass Personen, die mehr als 50.000 \$ verdienen, am ehesten an die Organisation spenden. Deshalb ist *CharityML* besonders daran interessiert, treffsicher vorherzusagen, wer mehr als 50.000 \$ verdient. Es scheint angemessen, die **Accuracy** als Metrik für die Evaluation der Leistung eines bestimmten Modells zu verwenden. Außerdem wäre es für *CharityML* schädlich, jemanden, der *nicht* mehr als 50.000 \$ verdient, als jemanden einzustufen, der es tut, denn die Organisation sucht Personen, die bereit sind zu spenden. Deshalb ist die Fähigkeit eines Modells, präzise vorherzusagen, wer mehr als 50.000 \$ verdient, *wichtiger* als sein **Recall** für diese Personen. Als Metrik, die sowohl Precision als auch Recall berücksichtigt, können wir den **F-beta-Score** verwenden:

$$ F_{\beta} = (1 + \beta^2) \cdot \frac{precision \cdot recall}{\left( \beta^2 \cdot precision \right) + recall} $$

Insbesondere wird bei $\beta = 0.5$ die Precision stärker gewichtet.

Ein Blick auf die Klassenverteilung (Personen, die höchstens 50.000 \$ verdienen, und Personen, die mehr verdienen) zeigt deutlich, dass die meisten Personen nicht mehr als 50.000 \$ verdienen. Das kann die **Accuracy** stark beeinflussen, denn wir könnten einfach sagen „*Diese Person verdient nicht mehr als 50.000 \$*“ und hätten meistens recht, ohne uns die Daten je angesehen zu haben! Eine solche Aussage gilt als **naiv**, da wir keinerlei Informationen berücksichtigt haben, die die Behauptung stützen. Es ist immer wichtig, die *naive Vorhersage* für deine Daten zu betrachten, um einen Benchmark dafür festzulegen, ob ein Modell gut abschneidet. Trotzdem wäre es sinnlos, diese Vorhersage zu verwenden: Wenn wir für alle Menschen vorhersagen würden, dass sie weniger als 50.000 \$ verdienen, würde *CharityML* keine Spendenden finden.

### Leistung des naiven Vorhersagemodells

Angenommen, wir wählen ein Modell, das immer vorhersagt, dass eine Person mehr als 50.000 \$ verdient: Welche Accuracy und welchen F-Score hätte dieses Modell auf diesem Datensatz? (Denk daran, was du am Anfang des Notebooks berechnet hast.)

In [ ]:
# TODO: Accuracy berechnen
accuracy = n_greater_50k / n_records

# TODO: Precision berechnen
precision = n_greater_50k / (n_greater_50k + n_at_most_50k)

# TODO: Recall berechnen
recall = n_greater_50k / (n_greater_50k + 0)

# TODO: F-Score mit der Formel oben für beta = 0.5 berechnen
fscore =  (1  + (0.5*0.5)) * ( precision * recall / (( 0.5*0.5 * (precision))+ recall))

# Ergebnisse ausgeben 
print ("Naive Predictor: [Accuracy score: {:.4f}, F-score: {:.4f}]".format(accuracy, fscore))

##  Modelle für überwachtes Lernen

###  Anwendung der Modelle

Jetzt wählen wir zwei der oben genannten Modelle für überwachtes Lernen aus, die für dieses Problem geeignet sind, und testen sie auf den Zensusdaten.

**Entscheidungsbäume**

 - Anwendung in der Praxis: Entscheidungsbäume und allgemein CART (Classification and Regression Trees) werden oft in der Finanzanalyse eingesetzt. Ein konkretes Beispiel dafür: anhand der bisherigen Wertentwicklung vorhersagen, welche Aktien gekauft werden sollten. [Referenz](https://ir.nctu.edu.tw/bitstream/11536/11962/1/000237645100007.pdf)
 - Stärken:
      - Können kategoriale und numerische Daten verarbeiten.
      - Brauchen wenig Datenvorverarbeitung und kommen mit Daten zurecht, die nicht normalisiert oder nicht passend für das Machine Learning kodiert wurden.
      - Einfach zu verstehen und zu interpretieren.
 - Schwächen:
     - Komplexe Entscheidungsbäume generalisieren schlecht auf die Daten und können zu Overfitting führen.
     - Instabil, da kleine Abweichungen in den Daten zu einem anderen Entscheidungsbaum führen können. Deshalb werden sie meist in einem Ensemble (wie Random Forests) eingesetzt, um Robustheit zu erreichen.
     - Können verzerrte Bäume erzeugen, wenn einige Klassen dominieren.
 - Eignung: Da ein Entscheidungsbaum sowohl numerische als auch kategoriale Daten verarbeiten kann, ist er ein guter Kandidat für unseren Fall (auch wenn die Vorverarbeitungsschritte einen möglichen Vorteil bereits aufheben könnten). Er ist außerdem leicht zu interpretieren, sodass wir wissen, was unter der Haube passiert, wenn wir die Ergebnisse interpretieren.



**Ensemble-Methoden: AdaBoost**

 - Anwendung in der Praxis: Ensemble-Methoden werden in Kaggle-Wettbewerben häufig eingesetzt, meist in der Bilderkennung. Ein Beispiel aus der Praxis für Adaboost ist die Objekterkennung in Bildern, z. B. das Erkennen der Spielenden während eines Basketballspiels. [Referenz](https://uni-obuda.hu/journal/Markoski_Ivankovic_Ratgeber_Pecev_Glusac_57.pdf)
 - Stärken:
     - Ensemble-Methoden, darunter Adaboost, sind robuster als einzelne Estimators und haben eine verbesserte Verallgemeinerbarkeit.
     - Einfache Modelle lassen sich zu einem komplexen Modell kombinieren, das rechnerisch schnell ist.
 - Schwächen:
     - Wenn wir einen verzerrten zugrunde liegenden Klassifikator haben, führt das zu einem verzerrten Boosting-Modell.
 - Eignung: Ensemble-Methoden gelten als hochwertige Klassifikatoren, und Adaboost ist einer der beliebtesten Boosting-Algorithmen. Außerdem haben wir in unserem Datensatz ein Klassenungleichgewicht, gegenüber dem Boosting robust sein könnte.

### Pipeline für Training und Vorhersage erstellen

Um die Leistung jedes ausgewählten Modells richtig zu evaluieren, ist es wichtig, dass du eine Pipeline für Training und Vorhersage erstellst, mit der du Modelle schnell und effektiv mit unterschiedlich großen Trainingsdaten trainieren und Vorhersagen für die Testdaten erstellen kannst.

In [ ]:
# TODO: Zwei Metriken aus sklearn importieren: fbeta_score und accuracy_score
from sklearn.metrics import fbeta_score, accuracy_score

# Funktion für Training und Evaluation definieren 
def train_predict(learner, sample_size, X_train, y_train, X_test, y_test): 
    '''
    inputs:
       - learner: Der Lernalgorithmus, der trainiert und für Vorhersagen genutzt wird
       - sample_size: Die Größe der Stichprobe (Anzahl), die aus dem Trainingsdatensatz gezogen wird
       - X_train: Trainingsdatensatz der Features
       - y_train: Trainingsdatensatz des Einkommens
       - X_test: Testdatensatz der Features
       - y_test: Testdatensatz des Einkommens
    '''
    
    results = {}
    
    # TODO: Learner per Slicing mit 'sample_size' auf den Trainingsdaten trainieren
    start = time() # Startzeit erfassen
    learner = learner.fit(X_train[:sample_size], y_train[:sample_size])
    end = time() # Endzeit erfassen
    
    # TODO: Trainingszeit berechnen
    results['train_time'] = end - start
        
    # TODO: Vorhersagen für den Testdatensatz erstellen,
    #       dann Vorhersagen für die ersten 300 Beobachtungen im Trainingsdatensatz erstellen
    start = time() # Startzeit erfassen
    predictions_test = learner.predict(X_test)
    predictions_train = learner.predict(X_train[:300])
    end = time() # Endzeit erfassen
    
    # TODO: Gesamte Vorhersagezeit berechnen
    results['pred_time'] = end - start
            
    # TODO: Accuracy für die ersten 300 Beobachtungen im Trainingsdatensatz berechnen
    results['acc_train'] = accuracy_score(y_train[:300], predictions_train)
        
    # TODO: Accuracy für den Testdatensatz berechnen
    results['acc_test'] = accuracy_score(y_test,predictions_test)
    
    # TODO: F-Score für die ersten 300 Beobachtungen im Trainingsdatensatz berechnen
    results['f_train'] = fbeta_score(y_train[:300], predictions_train, beta=0.5)
        
    # TODO: F-Score für den Testdatensatz berechnen
    results['f_test'] = fbeta_score(y_test, predictions_test, beta=0.5)
       
    # Erfolg
    print ("{} trained on {} samples.".format(learner.__class__.__name__, sample_size))
        
    # Ergebnisse zurückgeben
    return results

### Modellevaluation

Trainieren und testen wir die Modelle mit unterschiedlich großen Trainingsdatensätzen, um zu sehen, wie sich das auf ihre Laufzeit und ihre Vorhersageleistung auswirkt (sowohl auf dem Test- als auch auf dem Trainingsdatensatz).

In [ ]:
# TODO: Die zwei Modelle für überwachtes Lernen aus sklearn importieren
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

# TODO: Die zwei Modelle initialisieren, die Random States sind auf 101 gesetzt, damit wir wissen, wie wir das Modell später reproduzieren
clf_A = DecisionTreeClassifier(random_state=101)
clf_B = AdaBoostClassifier(random_state=101)

# TODO: Anzahl der Beobachtungen für 1%, 10% und 100% der Trainingsdaten berechnen
samples_1 = int(round(len(X_train) / 100))
samples_10 = int(round(len(X_train) / 10))
samples_100 = len(X_train)

# Ergebnisse der Learner sammeln
results = {}
for clf in [clf_A, clf_B]:
    clf_name = clf.__class__.__name__
    results[clf_name] = {}
    for i, samples in enumerate([samples_1, samples_10, samples_100]):
        results[clf_name][i] = \
        train_predict(clf, samples, X_train, y_train, X_test, y_test)

In [ ]:
# Visualisierung der Metriken für die zwei ausgewählten Modelle für überwachtes Lernen ausführen
vs.evaluate(results, accuracy, fscore)

Wir können die Werte aus den Visualisierungen oben auch ausgeben, um die Ergebnisse genauer zu untersuchen.

In [ ]:
#Werte ausgeben
for i in results.items():
    print (i[0])
    display(pd.DataFrame(i[1]).rename(columns={0:'1%', 1:'10%', 2:'100%'}))

Außerdem visualisieren wir die Confusion Matrix für die Ergebnisse.

In [ ]:
#Confusion Matrix für jeden Klassifikator visualisieren
from sklearn.metrics import confusion_matrix

for i,model in enumerate([clf_A,clf_B]):
    cm = confusion_matrix(y_test, model.predict(X_test))
    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] # Daten normalisieren

    # Als Heatmap anzeigen
    plt.figure(i)
    sns.heatmap(cm, annot=True, annot_kws={"size":30}, 
            cmap='Blues', square=True, fmt='.3f')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.title('Confusion matrix for:\n{}'.format(model.__class__.__name__));

Die Ergebnisse oben zeigen, dass AdaBoost von den beiden Modellen das geeignetere für unsere Aufgabe ist.

Vor allem schneidet es auf den Testdaten bei beiden Metriken besser ab: etwa 0,85 Accuracy und 0,70 F-Score gegenüber ungefähr 0,82 und 0,63 beim einzelnen Entscheidungsbaum. Außerdem lässt es sich schnell trainieren (etwa eine Sekunde auf dem vollständigen Trainingsdatensatz), es sollte also gut skalieren, wenn wir mehr Daten bekommen.

Standardmäßig verwendet Adaboost einen Entscheidungsstumpf, einen Entscheidungsbaum der Tiefe 1, als Basisklassifikator, der kategoriale und numerische Daten verarbeiten kann. Weak Learner lassen sich relativ schnell trainieren, daher ist die Größe des Datensatzes für den Algorithmus kein Problem.

### Aber wie funktioniert Adaboost?

1. Adaboost kombiniert mehrere einfache Learner (zum Beispiel Entscheidungsbäume) zu einem Ensemble aus Learnern, das vorhersagen kann, ob eine Person über 50k verdient oder nicht.

2. Jeder der Learner, in unserem Fall Entscheidungsbäume, wird mit den „Features“ erstellt, die wir über die Personen haben (z. B. Alter, Beruf, Bildung usw.), und bildet ein Regelwerk, mit dem sich das Einkommen einer Person vorhersagen lässt.

3. Während des Trainings, das mehrere Runden dauert, betrachtet der Adaboost-Algorithmus die Fälle, die er schlecht vorhergesagt hat, und priorisiert in der nächsten Trainingsrunde die korrekte Vorhersage dieser Fälle.

4. In jeder Runde findet das Modell den besten Learner (oder Entscheidungsbaum), den es ins Ensemble aufnimmt, und wiederholt den Vorgang für die festgelegte Anzahl an Runden oder bis wir die Vorhersagen nicht weiter verbessern können.

5. Alle Learner werden dann zu einem finalen Ensemble-Modell kombiniert, in dem jeder eine Stimme abgibt, um vorherzusagen, ob eine Person mehr als 50k verdient oder nicht. Normalerweise nehmen wir für die finale Vorhersage die Mehrheit der Stimmen.

6. Mit diesem Modell und den Zensusinformationen von Personen können wir dieselbe Information für eine potenzielle neue spendende Person vorhersagen, also ob sie mehr als 50K verdient oder nicht, und so entscheiden, wie wahrscheinlich es ist, dass sie für wohltätige Zwecke spendet.

## Unser Modell verbessern: Modell-Tuning

Hier findest du mehr Informationen zum Hyperparameter-Tuning mit Grid Search:

[Grid search - quick explanation and python implementation](https://medium.com/@elutins/grid-searching-in-machine-learning-quick-explanation-and-python-implementation-550552200596)

[Grid search for model tuning](https://towardsdatascience.com/grid-search-for-model-tuning-3319b259367e)

[Parameteroptimierung](https://www.datacamp.com/community/tutorials/parameter-optimisation-machine-learning-models)

Mit Grid Search (`GridSearchCV`) und verschiedenen Kombinationen aus Parametern und Werten können wir unser Modell für noch bessere Ergebnisse tunen.

Bei Adaboost tunen wir die Parameter n_estimators und learning rate sowie die Parameter des Basisklassifikators (denk daran, dass unser Basisklassifikator für das Adaboost-Ensemble ein Entscheidungsbaum ist!).

In [ ]:
# TODO: 'GridSearchCV', 'make_scorer' und alle weiteren benötigten Bibliotheken importieren
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer 

# TODO: Klassifikator initialisieren
clf = AdaBoostClassifier(estimator=DecisionTreeClassifier(), random_state=42) 

# TODO: Liste der Parameter erstellen, die du tunen möchtest
parameters = {'n_estimators':[50, 120],                
              'learning_rate':[0.1, 0.5, 1.],               
              'estimator__min_samples_split' : np.arange(2, 8, 2),               
              'estimator__max_depth' : np.arange(1, 4, 1)              
             } 

# TODO: Scoring-Objekt mit fbeta_score erstellen
scorer = make_scorer(fbeta_score,beta=0.5) 

# TODO: Grid Search auf dem Klassifikator mit 'scorer' als Scoring-Methode durchführen
grid_obj = GridSearchCV(clf, parameters, scoring=scorer, verbose=1, n_jobs=-1)

# TODO: Grid-Search-Objekt auf den Trainingsdaten trainieren und die optimalen Parameter finden
grid_fit = grid_obj.fit(X_train,y_train) 

# Estimator abrufen
best_clf = grid_fit.best_estimator_ 

# Vorhersagen mit dem nicht optimierten und dem Modell erstellen
predictions = (clf.fit(X_train, y_train)).predict(X_test)
best_predictions = best_clf.predict(X_test) 

# Scores vorher und nachher ausgeben
print("Unoptimized model\n------")
print("Accuracy score on testing data: {:.4f}".format(accuracy_score(y_test, predictions)))
print("F-score on testing data: {:.4f}".format(fbeta_score(y_test, predictions, beta = 0.5)))
print("\nOptimized Model\n------")
print("Final accuracy score on the testing data: {:.4f}".format(accuracy_score(y_test, best_predictions)))
print("Final F-score on the testing data: {:.4f}".format(fbeta_score(y_test, best_predictions, beta = 0.5)))
print(best_clf)

### Abschließende Modellevaluation

#### Ergebnisse:

|     Metrik     | Benchmark-Modell | Nicht optimiertes Modell | Optimiertes Modell |
| :------------: | :-----------------: | :---------------: | :-------------: |
| Accuracy-Score |         0,2478      |        0,8344     |      0,8680     |
| F-Score        |         0,2917      |        0,6630     |      0,7498     |


Das optimierte Modell hat eine Accuracy von 0,8680 und einen F-Score von 0,7498.

Diese Scores sind besser als beim nicht optimierten Modell und dabei deutlich besser als beim Benchmark-Modell.

----
## Feature Importance

Eine wichtige Aufgabe beim überwachten Lernen auf einem Datensatz wie den Zensusdaten, die wir hier untersuchen, ist herauszufinden, welche Features die größte Vorhersagekraft haben. Wenn wir uns auf den Zusammenhang zwischen nur wenigen entscheidenden Features und der Zielvariable konzentrieren, vereinfachen wir unser Verständnis des Phänomens, und das ist fast immer sinnvoll. Für dieses Projekt heißt das: Wir wollen eine kleine Anzahl von Features finden, die am stärksten vorhersagen, ob eine Person höchstens oder mehr als 50.000 \$ verdient.

###  Beobachtung zur Relevanz der Features

Wir wissen, dass es in den Zensusdaten für jede erfasste Person dreizehn verfügbare Features gibt.  Von diesen dreizehn Einträgen können wir raten, welche fünf Features für die Vorhersage am wichtigsten sein könnten. Probieren wir das aus.

Angenommen, die wichtigsten Features für die Vorhersage sind:

1. occupation: Verschiedene Berufe haben unterschiedliche Gehaltsstufen. Manche Berufe zahlen besser als andere.
2. education: Menschen mit einem höheren Bildungsabschluss sind besser für technischere/spezialisiertere Berufe gerüstet, die gut bezahlt werden.
3. age: Mit zunehmendem Alter häufen Menschen mehr Vermögen an.
4. workclass: Auch die Beschäftigungsklasse, der sie angehören, kann damit korrelieren, wie viel Geld sie verdienen.
5. hours-per-week: Wenn du mehr Stunden pro Woche arbeitest, verdienst du wahrscheinlich mehr.

Alle sind nach dem Einfluss geordnet, den sie auf das Einkommen einer Person haben könnten. Der Beruf steht auf Platz eins, weil verschiedene Berufe unterschiedliche Gehaltsstufen haben. Menschen mit höherer Bildung verdienen wahrscheinlich besser.

Sehen wir uns jetzt an, wie unser Algorithmus die Features nach ihrer Feature Importance ordnet.

### Feature Importance extrahieren

In [ ]:
# TODO: Ein Modell für überwachtes Lernen importieren, das 'feature_importances_' hat

# TODO: Das Modell für überwachtes Lernen auf dem Trainingsdatensatz trainieren 
model = AdaBoostClassifier().fit(X_train,y_train)

# TODO: Feature Importances extrahieren
importances = model.feature_importances_

# Darstellen
vs.feature_plot(importances, X_train, y_train)

Von den fünf Features, die wir im vorherigen Abschnitt vermutet haben, sind 3, nämlich Age, hours per week und education-num (ein numerisches Label für die Bildung), in der Liste der Features enthalten, die Adaboost für am wichtigsten hält, wenn auch in anderer Rangfolge.

Zwei wichtige Features, capital-gain und capital-loss, habe ich nicht berücksichtigt, teils weil ich nicht verstanden hatte, was sie bedeuten. Nachdem ich recherchiert habe, was sie bedeuten (Gewinn oder Verlust aus dem Verkauf von Vermögenswerten/Eigentum), ist es einleuchtend, dass diese Features wichtig sind. Personen, die Gewinne aus dem Verkauf von Vermögenswerten erzielt haben, verdienen definitiv eher mehr, während Personen mit Verlusten wahrscheinlich ein niedrigeres Gesamteinkommen hatten.




### Feature-Auswahl

Eine interessante Frage ist hier: Wie schneidet ein Modell ab, wenn wir nur eine Teilmenge aller verfügbaren Features in den Daten verwenden? Mit weniger Features für das Training erwarten wir, dass Trainings- und Vorhersagezeit deutlich sinken, auf Kosten der Leistungsmetriken. In der Visualisierung oben sehen wir, dass die fünf wichtigsten Features mehr als die Hälfte der Feature Importance **aller** Features in den Daten ausmachen. Das deutet darauf hin, dass wir versuchen können, *den Feature-Raum zu verkleinern* und die Informationen zu vereinfachen, die das Modell zum Lernen braucht.

Sehen wir uns an, wie ein Modell abschneidet, das nur auf den ausgewählten Features trainiert wurde.

In [ ]:
# Funktion zum Klonen eines Modells importieren
from sklearn.base import clone

# Feature-Raum verkleinern
X_train_reduced = X_train[X_train.columns.values[(np.argsort(importances)[::-1])[:5]]]
X_test_reduced = X_test[X_test.columns.values[(np.argsort(importances)[::-1])[:5]]]

# Das „beste“ Modell aus dem Grid Search von vorhin trainieren
clf = (clone(best_clf)).fit(X_train_reduced, y_train)

# Neue Vorhersagen erstellen
reduced_predictions = clf.predict(X_test_reduced)

# Scores des finalen Modells für beide Versionen der Daten ausgeben
print("Final Model trained on full data\n------")
print("Accuracy on testing data: {:.4f}".format(accuracy_score(y_test, best_predictions)))
print("F-score on testing data: {:.4f}".format(fbeta_score(y_test, best_predictions, beta = 0.5)))
print("\nFinal Model trained on reduced data\n------")
print("Accuracy on testing data: {:.4f}".format(accuracy_score(y_test, reduced_predictions)))
print("F-score on testing data: {:.4f}".format(fbeta_score(y_test, reduced_predictions, beta = 0.5)))

### Auswirkungen der Feature-Auswahl

Auf einem reduzierten Datensatz sind Accuracy und F-Score des finalen Modells noch sehr gut mit den Ergebnissen auf dem vollständigen Datensatz vergleichbar.

Die Accuracy ist um ~2,6% niedriger, der F-Score um ~5%. Auch wenn Adaboost relativ schneller ist als einer der anderen Klassifikatoren, die wir ausprobiert haben, würde ich trotzdem ein Training auf den reduzierten Daten (nach Features) in Betracht ziehen, wenn die Trainingszeit eine Rolle spielt und wir mehr Trainingspunkte verarbeiten müssen. Diese Entscheidung hängt auch davon ab, wie wichtig Accuracy und F-Scores sind (oder ob der F-Score wichtiger ist als die Accuracy, da sein Rückgang größer ist als der Rückgang der Accuracy), um hier eine endgültige Entscheidung zu treffen.

## Zusammenfassung

In diesem Notebook hast du:

- die Zensusdaten vorverarbeitet: die schiefen Features log-transformiert, die kategorialen Features per One-Hot Encoding kodiert und die numerischen Spalten skaliert.
- eine naive Baseline erstellt und den F-beta-Score (beta = 0.5) als Metrik übernommen, da er bei einer unbalancierten Zielvariable die Precision stärker gewichtet als den Recall.
- einen Entscheidungsbaum und AdaBoost verglichen und AdaBoost als besten Kandidaten ausgewählt.
- AdaBoost mit `GridSearchCV` getunt und den Trade-off gemessen, den ein Training nur auf den fünf wichtigsten Features mit sich bringt.

## Quellen & weiterführende Literatur

- [**Scikit-learn: AdaBoostClassifier**](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html): API-Referenz für das hier ausgewählte Boosting-Modell.
- [**Scikit-learn: GridSearchCV**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html): API-Referenz für das kreuzvalidierte Grid Search, mit dem das Modell getunt wird.
- [**Scikit-learn: Ensemble Methods**](https://scikit-learn.org/stable/modules/ensemble.html): Benutzerhandbuch zu Boosting, Bagging, Voting und Stacking.
- [**Scikit-learn: Metrics and Scoring**](https://scikit-learn.org/stable/modules/model_evaluation.html): Wie der F-beta-Score und die anderen Metriken definiert sind.